# Notebook 04 — Churn Dashboard Final
**Tujuan:** Layout satu halaman siap presentasi client — KPI cards + 5 chart utama.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import json
from pathlib import Path
from sklearn.metrics import roc_curve, roc_auc_score

OUT = Path('../output')

BLUE    = '#2563EB'
LIGHT   = '#93C5FD'
LIGHTER = '#DBEAFE'
RED     = '#DC2626'
AMBER   = '#F59E0B'
GRAY    = '#6B7280'
BG      = '#F8FAFC'

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.facecolor': BG,
    'axes.facecolor': '#FFFFFF',
})

df      = pd.read_parquet(OUT / 'df_clean.parquet')
results = json.loads((OUT / 'model_results.json').read_text())
print('Data loaded.')

In [ ]:
# ── Pre-compute ─────────────────────────────────────────────────────────────
churn_rate   = df['Churn'].mean()
at_risk      = results['business_impact']['at_risk_customers']
rev_risk     = results['business_impact']['revenue_at_risk_monthly_usd']
auc_rf       = results['random_forest']['auc']
recall_rf    = results['random_forest']['recall']

# Contract churn rate
cc = df.groupby('Contract')['Churn'].mean().sort_values(ascending=False)

# Tenure segment churn rate
seg_order = ['New (<12 bln)', 'Mid (12-24 bln)', 'Loyal (>24 bln)']
tc = df.groupby('tenure_segment')['Churn'].mean().reindex(seg_order)

# Feature importance
fi = pd.Series(results['feature_importance_top10']).sort_values()

# Risk segment
seg_sum = (
    df.groupby('risk_segment')
    .agg(count=('Churn','count'), churn_rate=('Churn','mean'))
    .reindex(['High Risk','Mid Risk','Low Risk'])
    .reset_index()
)

print('Churn rate:', f'{churn_rate:.1%}')
print('AUC RF    :', auc_rf)

In [ ]:
# ── Build Dashboard ─────────────────────────────────────────────────────────
fig = plt.figure(figsize=(20, 14), facecolor=BG)
fig.suptitle('Telco Customer Churn — Analysis & Prediction Dashboard',
             fontsize=20, fontweight='bold', y=0.98, color='#1E3A5F')
fig.text(0.5, 0.955, 'IBM Telco Dataset  |  7,043 Pelanggan  |  Model: Random Forest',
         ha='center', fontsize=11, color=GRAY)

gs = gridspec.GridSpec(3, 4, figure=fig,
    height_ratios=[0.18, 0.42, 0.40],
    hspace=0.48, wspace=0.38,
    top=0.93, bottom=0.06, left=0.05, right=0.97)

# ── ROW 0: KPI Cards ────────────────────────────────────────────────────────
kpis = [
    ('Churn Rate',          f'{churn_rate:.1%}',        RED),
    ('Pelanggan Berisiko',  f'{at_risk:,}',             AMBER),
    ('Revenue at Risk/bln', f'${rev_risk/1000:.0f}K',   RED),
    ('Model AUC (RF)',      f'{auc_rf:.3f}',            BLUE),
]
for col, (label, value, color) in enumerate(kpis):
    ax = fig.add_subplot(gs[0, col])
    ax.axis('off')
    rect = mpatches.FancyBboxPatch((0,0), 1, 1,
        boxstyle='round,pad=0.05', facecolor='#FFFFFF',
        edgecolor=LIGHTER, linewidth=1.5,
        transform=ax.transAxes, clip_on=False)
    ax.add_patch(rect)
    ax.text(0.5, 0.65, value, ha='center', va='center',
            fontsize=18, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.5, 0.22, label, ha='center', va='center',
            fontsize=10, color=GRAY, transform=ax.transAxes)

# ── ROW 1, col 0-1: Churn by Contract ───────────────────────────────────────
ax_cc = fig.add_subplot(gs[1, :2])
bar_c = [RED if v == cc.max() else LIGHT for v in cc.values]
ax_cc.bar(cc.index, cc.values * 100, color=bar_c, width=0.5)
ax_cc.set_ylabel('Churn Rate (%)')
ax_cc.yaxis.set_major_formatter(mticker.PercentFormatter())
ax_cc.set_title('Churn Rate per Jenis Kontrak', fontweight='bold', fontsize=12)
for i, v in enumerate(cc.values):
    ax_cc.text(i, v * 100 + 0.5, f'{v:.1%}', ha='center', fontweight='bold', fontsize=10)

# ── ROW 1, col 2-3: Churn by Tenure Segment ─────────────────────────────────
ax_tc = fig.add_subplot(gs[1, 2:])
seg_colors = [RED, AMBER, BLUE]
ax_tc.bar(tc.index, tc.values * 100, color=seg_colors, width=0.5)
ax_tc.set_ylabel('Churn Rate (%)')
ax_tc.yaxis.set_major_formatter(mticker.PercentFormatter())
ax_tc.set_title('Churn Rate per Segmen Tenure', fontweight='bold', fontsize=12)
for i, v in enumerate(tc.values):
    ax_tc.text(i, v * 100 + 0.5, f'{v:.1%}', ha='center', fontweight='bold', fontsize=10)

# ── ROW 2, col 0-1: Feature Importance ──────────────────────────────────────
ax_fi = fig.add_subplot(gs[2, :2])
fi_colors = [BLUE if i == len(fi)-1 else LIGHT for i in range(len(fi))]
ax_fi.barh(fi.index, fi.values, color=fi_colors)
ax_fi.set_xlabel('Importance Score')
ax_fi.set_title('Top 10 Faktor Prediksi Churn (Random Forest)', fontweight='bold', fontsize=12)
ax_fi.tick_params(axis='y', labelsize=8)
for i, v in enumerate(fi.values):
    ax_fi.text(v + 0.001, i, f'{v:.3f}', va='center', fontsize=7.5)

# ── ROW 2, col 2: Risk Segment Churn Rate ───────────────────────────────────
ax_seg = fig.add_subplot(gs[2, 2])
ax_seg.bar(seg_sum['risk_segment'], seg_sum['churn_rate'] * 100,
           color=[RED, AMBER, BLUE], width=0.5)
ax_seg.set_ylabel('Churn Rate (%)')
ax_seg.yaxis.set_major_formatter(mticker.PercentFormatter())
ax_seg.set_title('Churn Rate\nper Risk Segment', fontweight='bold', fontsize=11)
ax_seg.tick_params(axis='x', labelsize=8)
for i, v in enumerate(seg_sum['churn_rate']):
    ax_seg.text(i, v * 100 + 0.5, f'{v:.1%}', ha='center', fontweight='bold', fontsize=9)

# ── ROW 2, col 3: ROC Curve ─────────────────────────────────────────────────
ax_roc = fig.add_subplot(gs[2, 3])
# Rebuild prediksi untuk ROC (hanya test set)
# Catatan: ROC curve menggunakan churn_prob yang sudah dihitung di nb03
# Untuk dashboard, tampilkan AUC sebagai annotation saja
ax_roc.text(0.5, 0.55, f'{auc_rf:.3f}', ha='center', va='center',
            fontsize=36, fontweight='bold', color=BLUE, transform=ax_roc.transAxes)
ax_roc.text(0.5, 0.35, 'ROC-AUC Score', ha='center', va='center',
            fontsize=11, color=GRAY, transform=ax_roc.transAxes)
ax_roc.text(0.5, 0.22, f'Recall Churn: {recall_rf:.1%}', ha='center', va='center',
            fontsize=10, color=RED, transform=ax_roc.transAxes)
ax_roc.axis('off')
rect2 = mpatches.FancyBboxPatch((0,0), 1, 1,
    boxstyle='round,pad=0.05', facecolor='#FFFFFF',
    edgecolor=LIGHTER, linewidth=1.5,
    transform=ax_roc.transAxes, clip_on=False)
ax_roc.add_patch(rect2)
ax_roc.text(0.5, 0.78, 'Model Performance', ha='center', va='center',
            fontsize=11, fontweight='bold', color='#1E3A5F', transform=ax_roc.transAxes)
ax_roc.text(0.5, 0.55, f'{auc_rf:.3f}', ha='center', va='center',
            fontsize=36, fontweight='bold', color=BLUE, transform=ax_roc.transAxes)
ax_roc.text(0.5, 0.35, 'ROC-AUC', ha='center', va='center',
            fontsize=11, color=GRAY, transform=ax_roc.transAxes)
ax_roc.text(0.5, 0.18, f'Recall: {recall_rf:.1%}', ha='center', va='center',
            fontsize=13, fontweight='bold', color=RED, transform=ax_roc.transAxes)

# ── Footer ──────────────────────────────────────────────────────────────────
fig.text(0.5, 0.015,
    'Prepared by Ray  |  Tools: Python, Pandas, Scikit-learn, Matplotlib  |  Source: IBM Telco Customer Churn Dataset (Kaggle)',
    ha='center', fontsize=8.5, color=GRAY)

plt.savefig(OUT / 'dashboard.png', dpi=150, bbox_inches='tight', facecolor=BG)
print('Saved: output/dashboard.png')
plt.show()

## Executive Summary

### Temuan Utama
| Dimensi | Insight |
|---|---|
| **Churn Rate** | ~26.5% — jauh di atas rata-rata industri telco 1-3%/bulan; urgen untuk retensi proaktif |
| **Kontrak** | Month-to-month churn >40%; two-year contract <5% — tipe kontrak adalah prediktor tunggal paling kuat |
| **Tenure** | Pelanggan baru (<12 bulan) churn 3x lebih tinggi vs loyal (>24 bulan) — onboarding 30 hari pertama kritis |
| **Revenue at Risk** | Model mengidentifikasi pelanggan High Risk — intervensi awal bisa menyelamatkan revenue bulanan signifikan |
| **Model Performance** | Random Forest AUC >0.83; Recall ~78% — 8 dari 10 churner berhasil diidentifikasi sebelum pergi |

### Business Recommendation
1. **Prioritas:** Fokus intervensi pada segmen High Risk dengan tenure <12 bulan dan kontrak month-to-month.
2. **Onboarding:** Program "30-day success" untuk pelanggan baru — engagement di bulan pertama menurunkan churn 20-30%.
3. **Upsell Kontrak:** Tawarkan diskon 15-20% untuk upgrade ke annual/two-year contract — LTV meningkat 3x.

> **Relevansi Indonesia:** Telkomsel/XL/Indosat: pelanggan prepaid bersifat month-to-month by default — churn equivalent 3-5%/bulan. E-wallet linkage (GoPay, OVO, DANA) terbukti menurunkan churn 15-20% karena meningkatkan switching cost.
